In [1]:
import matplotlib.pyplot as plt
#import numpy as np
import pandas as pd
import os
import shutil
import re
import seaborn as sns
from itertools import combinations
import numpy as np
from pymatgen.core.structure import Structure

In [2]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [3]:
#Done
def get_elneg(atom_type):
    from pymatgen.core.periodic_table import Element

    # Example: Get electronegativity of Oxygen
    elneg = Element(atom_type).X
    return elneg

In [5]:
#Done
def vasp_to_coordinate(coord_line, cell_matrix, atom_type, fractional =False):
    coord_list = coord_line.split()[0:3]
    #atom_type = coord_line.split()[0]
    for n in range(len(coord_list)):
        #coord_list[n] = re.sub("d","0",coord_list[n])
        coord_list[n] = float(coord_list[n])
    coord_array = np.array(coord_list).reshape(1,3)
    if fractional == False:
        coord_array_ang = np.dot(coord_array,cell_matrix)
        return (atom_type,coord_array_ang) 
    else:
        return (atom_type,coord_array)

In [6]:
#Done
decimal = 4
def TOTEN_vasp(outcar_path, decimal = decimal):
    outcar = open(outcar_path, 'r')
    outcar_lines = outcar.readlines()
    for n_line in range(len(outcar_lines)):
        if 'TOTEN' in outcar_lines[n_line]:
            TOTEN = round(float(outcar_lines[n_line].split()[4]),decimal)
    return TOTEN

In [7]:
#Done
def countX(lst, x):
    count = 0
    for ele in lst:
        if (ele == x):
            count = count + 1
    return count

In [8]:
#Done
def two_atoms_dist(atom1_coord, atom2_coord, decimal = decimal):
    return round(np.linalg.norm(atom1_coord - atom2_coord),decimal)

In [9]:
#Done
def find_element(list_, line):
    final_result = False
    for element in list_:
        if element in line:
            result = True
        else:
            result = False
        final_result = final_result or result
    return final_result
            

# INCAR Files 

In [10]:
from pymatgen.io.vasp.inputs import Potcar

def get_valence_electrons_from_potcar_pymatgen(potcar_file):
    """
    Reads a POTCAR file using pymatgen and extracts the number of valence electrons for each element.

    Args:
        potcar_file (str): Path to the POTCAR file.

    Returns:
        dict: A dictionary where keys are element names and values are the number of valence electrons.
    """
    try:
        # Use pymatgen's Potcar class to parse the POTCAR file
        potcar = Potcar.from_file(potcar_file)
        
        # Extract element names and their ZVAL values
        valence_electrons = {entry.element: entry.nelectrons for entry in potcar}
        
        return valence_electrons
    except FileNotFoundError:
        raise FileNotFoundError(f"The file {potcar_file} does not exist.")
    except Exception as e:
        raise RuntimeError(f"An error occurred while parsing the POTCAR file: {e}")


In [11]:
#test
valence_electrons = get_valence_electrons_from_potcar_pymatgen('./test_repeat_structure/0_VASP/SDA_test/0_test/POTCAR.surf')
valence_electrons

{'Co': 9.0, 'Ni': 10.0}

In [12]:
#Done

#In this script since we are working mainly with ACF.dat file, it works for all different versions of POSCAR 
#and qs_input files. 
# For vasp inputs you can use get_valence_electrons_from_pymatgen function to create the dictionary 
#for valence electrons


valence_electrons = {'Ag':19.0,
                     'Au':11.0,
                     'Co':9.0,
                     'Fe':16.0,
                     'Ru':16.0,
                     'Pd':16.0,
                     'Pt':16.0,
                     'Cu':19.0,
                     'Ni':10.0,
                      'N':5.0,
                      'H':1}

def bader_df(acf_dat, #path to the acf.dat file
             input_path, #path to the input file
             input_type = 'qs', #'vasp' or 'qs'
             valence_dict = valence_electrons, #dictionary which lists the valence electrons
             slab = True, #if the structure is slab or bulk to specify surface atoms
             atom_types_line_num = 5,  #The line number in 'POSCAR' which specify the atom types
             type_num_line_num = 6, #The line number in 'POSCAR' which specify the number of atoms for each type
             z_threshhold = 1.0, #What is the maximum value of difference between the z parameter of surface layers for example they are all in range 14.5, 15,5 
             moleclue_exist = False,
             num_atom_mol = 2):

    acf_file = open(acf_dat, 'r')
    acf_lines = acf_file.readlines()[2:-4]
    
    acf_file.close()
    col_names=['#','x','y','z','Charge','Min_dist','atomic_volume']
    acf_df = pd.DataFrame()
    for i in range(len(col_names)):
        acf_df[col_names[i]] = [float(re.findall(r'-*\d+\.*\d*', line.split()[i])[0]) for line in acf_lines]
    acf_df.drop('#',axis=1,inplace=True)
    
    input_file = open(input_path, 'r')
    input_file_lines = input_file.readlines() 
    input_file.close()
    if input_type == 'qs':
        for line_num in range(len(input_file_lines)):
            line = input_file_lines[line_num]
            if 'nat' in line:
                nat = int(re.findall(r'\d+', line)[0])
            if 'ATOMIC_POSITIONS' in line:
                start_coord = line_num + 1
                #end_coord = start_coord + nat
                break
        end_coord = start_coord + nat
        coordinate_lines = input_file_lines[start_coord: end_coord] 
        atoms_list = []
        Zs = list(acf_df['z'])
        #Zs = []
        for line in coordinate_lines:
            atom_type = line.split()[0]
        #    z = line.split()[3]
        #    z_float = float(re.sub("d","0",z))
            atoms_list.append(atom_type)
        #    Zs.append(z_float)
        acf_df['z_alat'] = Zs
        #print(acf_df['z_alat'])
        if moleclue_exist == True:
            z_max = sorted(Zs,reverse = True)[num_atom_mol]
        elif moleclue_exist == False:
            z_max = sorted(Zs,reverse = True)[0]
        #print(z_max)
        
            
        
    elif input_type == 'vasp':
        atom_types = input_file_lines[atom_types_line_num].split()
        type_num = [int(i) for i in input_file_lines[type_num_line_num].split()]
        atoms_list = ''
        for i in range(0,len(atom_types)):
            atoms_list = atoms_list + type_num[i] * (atom_types[i] + ' ')
        atoms_list = atoms_list.split()
        Zs = list(acf_df['z'])
        acf_df['z_alat'] = Zs
        if moleclue_exist == True:
            z_max = sorted(Zs,reverse = True)[num_atom_mol]
        elif moleclue_exist == False:
            z_max = sorted(Zs,reverse = True)[0]
        #print(z_max)
    
    #diff_z = [abs(i-z_max) for i in Zs]    
    #for i in diff_z:
        #print(i)          
        
        
    
    acf_df['atom_type'] = atoms_list
    #print(atoms_list)
    acf_df = acf_df[['atom_type','x', 'y', 'z', 'Charge', 'Min_dist', 'atomic_volume','z_alat']]
    
    if slab == True:
        acf_df['surface_atom'] = np.where(acf_df['z_alat'] > z_max - z_threshhold,1,0)
        
    acf_df.drop('z_alat',axis=1,inplace=True)    
    valence_list = [valence_dict.get(i) for i in atoms_list] 
    acf_df['bader_charge'] = valence_list - acf_df['Charge']
    #acf_df.index = acf_df.index + 1
    
    return acf_df
    
#bader_df(acf_dat = '0_test_bader/ACF_original.dat', 
         #input_type = 'vasp',
         #input_path = '0_test_bader/CONTCAR')        
    
        
        
        
            
        
#This script has a problem it is not worked on vasp input when molecule exists, only qs is fixed for
#that condition, you have to consider adding exactly the same z_alat column using vasp poscar.


In [13]:
#Test qs
bader_df('./0_test_bader/ACF_original.dat', 
         './0_test_bader/input.in',
             input_type = 'qs',
             valence_dict = valence_electrons,
             slab = True,
             atom_types_line_num = 5, 
             type_num_line_num = 6,
             z_threshhold = 1.0)

,atom_type,x,y,z,Charge,Min_dist,atomic_volume,surface_atom,bader_charge
0,Cu,19.2343,0.0253,4.1746,19.0128,2.0694,380.7495,0,-0.0128
1,Ni,4.7013,0.0458,4.2969,18.0627,2.0256,305.6438,0,-8.0627
2,Cu,-0.0882,16.7327,4.1029,19.0112,2.1353,414.8718,0,-0.0112
3,Cu,14.3793,0.0475,4.0921,19.0160,2.1208,425.2662,0,-0.0160
4,Ni,16.7295,4.3127,4.2657,18.0381,2.0000,344.0589,0,-8.0381
5,Cu,2.2743,4.1930,4.1869,19.0169,2.0806,387.9776,0,-0.0169
6,Cu,7.2148,4.1865,4.1348,19.0099,2.0975,419.0325,0,-0.0099
7,Ni,12.0526,4.3116,4.2890,18.0440,2.0641,326.5942,0,-8.0440
8,Ni,14.4146,8.3093,4.2828,18.0256,2.0810,337.6821,0,-8.0256
9,Ni,-0.1948,8.4080,4.2813,18.0561,2.0167,331.3676,0,-8.0561


In [14]:
#test vasp
input_path = './test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.surf'
acf_dat = './test_repeat_structure/0_VASP/SDA_test/0_test/ACF.dat'
bader_df(acf_dat,
         input_path,
         input_type = 'vasp',
         valence_dict = valence_electrons,
         slab = True,
         atom_types_line_num = 5, 
         type_num_line_num = 6,
         z_threshhold = 1.0)

,atom_type,x,y,z,Charge,Min_dist,atomic_volume,surface_atom,bader_charge
0,Co,0.6309,1.7728,1.8977,8.9813,1.0668,48.3972,0,0.0187
1,Co,3.0651,1.7728,1.8977,8.9677,1.0504,49.6328,0,0.0323
2,Co,5.5664,1.7728,1.8977,8.9813,1.0668,48.3972,0,0.0187
3,Co,8.0006,1.7728,1.8977,8.9677,1.0504,49.6328,0,0.0323
4,Co,-0.6197,3.9390,1.8977,8.9778,1.0512,48.4449,0,0.0222
5,Co,4.3157,3.9390,1.8977,8.9778,1.0512,48.4449,0,0.0222
6,Co,-1.8368,6.0471,1.8977,8.9813,1.0668,48.3972,0,0.0187
7,Co,0.5974,6.0471,1.8977,8.9677,1.0504,49.6328,0,0.0323
8,Co,3.0987,6.0471,1.8977,8.9813,1.0668,48.3972,0,0.0187
9,Co,5.5328,6.0471,1.8977,8.9677,1.0504,49.6328,0,0.0323


In [15]:
#Done
#This was too manual but if with same function in the next block you faced a problem go back to this one again
import numpy as np
def read_CONTCAR_old(contcar_path, 
                 output_type = 'list',  # 'list' or df
                 coord_start_line_num = 9,
                 cell_dm_ang_line_num = 1,
                 cell_pars_line_num = 2, 
                 atom_types_line_num = 5, 
                 type_num_line_num = 6,
                 fractional=False):
    
    def vasp_to_coordinate(coord_line, cell_matrix, atom_type, fractional =False):
        coord_list = coord_line.split()[0:3]
        #atom_type = coord_line.split()[0]
        for n in range(len(coord_list)):
            #coord_list[n] = re.sub("d","0",coord_list[n])
            coord_list[n] = float(coord_list[n])
        coord_array = np.array(coord_list).reshape(1,3)
        if fractional == False:
            coord_array_ang = np.dot(coord_array,cell_matrix)
            return (atom_type,coord_array_ang) 
        else:
            return (atom_type,coord_array)
    
    all_coords = []
    #Extracting information from final CONTCAR
    contcar = open(contcar_path,'r')
    contcar_lines = contcar.readlines()
    contcar.close()

    ##cell matrix
    cell_dm_ang = float(contcar_lines[cell_dm_ang_line_num].split()[0])
    list_cell_matrix = [float(i) for i in contcar_lines[cell_pars_line_num].split() 
                        + contcar_lines[cell_pars_line_num + 1].split() 
                        + contcar_lines[cell_pars_line_num + 2].split()]
    cell_matrix = np.array(list_cell_matrix).reshape(3,3) * cell_dm_ang 

    ##generations of atom types list
    atom_types = contcar_lines[atom_types_line_num].split()
    type_num = [int(i) for i in contcar_lines[type_num_line_num].split()]
    atoms_list = ''
    for i in range(0,len(atom_types)):
        atoms_list = atoms_list + type_num[i] * (atom_types[i] + ' ')
    atoms_list = atoms_list.split()

    ##Reading cooordinates oof CONTCAR
    #The line number in the contcar where the first coordinates is written (start counting from zero not one)
    coordinates = contcar_lines[coord_start_line_num : coord_start_line_num + sum(type_num)]
    for num in range(len(coordinates)):    
        coordinate = vasp_to_coordinate(coordinates[num],cell_matrix, atoms_list[num],fractional=fractional)
        all_coords.append(coordinate)
        
    df = pd.DataFrame()
    atom_type = [i[0] for i in all_coords]
    x = [i[1][0][0] for i in all_coords]
    y = [i[1][0][1] for i in all_coords]
    z = [i[1][0][2] for i in all_coords]
    df['atom_type'] = atom_type
    df['x'] = x
    df['y'] = y
    df['z'] = z
    
    if output_type == 'list':
        return all_coords
    else:
        return df


In [16]:
import numpy as np
import pandas as pd
from pymatgen.core import Structure

def read_CONTCAR(file_path, fractional=False, output_type='df'):
    """
    Reads a CONTCAR file and returns atomic coordinates as a list or pandas DataFrame.

    Parameters:
        file_path (str): Path to the CONTCAR file.
        fractional (bool): If True, return fractional coordinates. If False, return real coordinates.
        output_type (str): 'list' for a list of coordinates (as 3×1 NumPy arrays), 'df' for a pandas DataFrame.

    Returns:
        list or pd.DataFrame: Atomic coordinates in requested format.
    """
    # Load structure using pymatgen
    structure = Structure.from_file(file_path)
    
    # Get atomic symbols
    atom_types = [site.species_string for site in structure]
    
    # Get coordinates (fractional or real)
    coordinates = structure.frac_coords if fractional else structure.cart_coords

    # If output_type is 'list', return a list with coordinates as (atom_type, 3×1 NumPy array)
    if output_type == 'list':
        return [(atom, np.array([[x], [y], [z]])) for atom, (x, y, z) in zip(atom_types, coordinates)]
    
    # If output_type is 'df', return a pandas DataFrame
    elif output_type == 'df':
        df = pd.DataFrame(coordinates, columns=['x', 'y', 'z'])
        df.insert(0, 'atom_type', atom_types)
        return df
    
    else:
        raise ValueError("Invalid output_type. Choose 'list' or 'df'.")



In [17]:
#test
read_CONTCAR_old('./test_repeat_structure/1_QS/1_acf_qs_to_vasp/POSCAR', 
                 output_type = 'list',  # 'list' or df
                 coord_start_line_num = 9,
                 cell_dm_ang_line_num = 1,
                 cell_pars_line_num = 2, 
                 atom_types_line_num = 5, 
                 type_num_line_num = 6,
            fractional=False)

[('Cu', array([[1.02176869e+01, 9.25331456e-03, 2.17263780e+00]])),
 ('Cu', array([[-2.66864564e-03,  8.85422663e+00,  2.12101451e+00]])),
 ('Cu', array([[6.38641664, 3.69147924, 6.24581125]])),
 ('Au', array([[2.56133234e+00, 4.24147146e-03, 8.30731071e+00]])),
 ('Ni', array([[7.66956997, 2.96948162, 4.21646253]])),
 ('Ni', array([[-1.29185966,  5.13710376,  4.24359336]]))]

In [18]:
import numpy as np
import pandas as pd
from pymatgen.core import Structure

def read_CONTCAR(file_path, fractional=False, output_type='df'):
    """
    Reads a CONTCAR file and returns atomic coordinates as a list or pandas DataFrame.

    Parameters:
        file_path (str): Path to the CONTCAR file.
        fractional (bool): If True, return fractional coordinates. If False, return real coordinates.
        output_type (str): 'list' for a list of coordinates (as 3×1 NumPy arrays), 'df' for a pandas DataFrame.

    Returns:
        list or pd.DataFrame: Atomic coordinates in requested format.
    """
    # Load structure using pymatgen
    structure = Structure.from_file(file_path)
    
    # Get atomic symbols
    atom_types = [site.species_string for site in structure]
    
    # Get coordinates (fractional or real)
    coordinates = structure.frac_coords if fractional else structure.cart_coords

    # If output_type is 'list', return a list with coordinates as (atom_type, 3×1 NumPy array)
    if output_type == 'list':
        return [(atom, np.array([[x], [y], [z]])) for atom, (x, y, z) in zip(atom_types, coordinates)]
    
    # If output_type is 'df', return a pandas DataFrame
    elif output_type == 'df':
        df = pd.DataFrame(coordinates, columns=['x', 'y', 'z'])
        df.insert(0, 'atom_type', atom_types)
        return df
    
    else:
        raise ValueError("Invalid output_type. Choose 'list' or 'df'.")



In [19]:
#test
read_CONTCAR('./test_repeat_structure/1_QS/1_acf_qs_to_vasp/POSCAR', 
                 output_type = 'df',  # 'list' or df
            fractional=True)

,atom_type,x,y,z
0,Cu,0.999837,0.001045,0.093053
1,Cu,0.499704,0.999930,0.090842
2,Cu,0.833051,0.416888,0.267505
3,Au,0.250744,0.000479,0.355798
4,Ni,0.917778,0.335351,0.180589
5,Ni,0.163726,0.580146,0.181751


In [20]:
def expand_struct_vasp(input_path, output_path, supercell_size):
    from pymatgen.core.structure import Structure
    structure = Structure.from_file(input_path)
    structure.make_supercell(supercell_size)
    structure.to(fmt = 'POSCAR',filename = output_path)
    print('Structure is Expanded')

In [21]:
expand_struct_vasp('./test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.surf',
                  './test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.surf221',
                  [2,2,1])

Structure is Expanded


In [22]:
#Done
def expand_charge_file_VASP(acf_dat,
                            input_path,
                            potcar_path,
                            input_type = 'vasp',
                            supercell_size = [2,2,1]):
    """
    Expands a VASP structure by creating a supercell, extracts valence electrons from a POTCAR file, 
    and processes charge information from an acf.dat file.

    Args:
        acf_dat (str): Path to the acf.dat file containing charge information, unexpanded.
        input_path (str): Path to the input POSCAR file (VASP structure file), unexpanded.
        potcar_path (str): Path to the POTCAR file, used to extract valence electron information.
        input_type (str, optional): Specifies the type of input format. Default is 'vasp'.
        supercell_size (list, optional): Defines the expansion size for creating a supercell. Default is [2,2,1].

    Returns:
        pd.DataFrame: Expanded Bader charge data including Cartesian coordinates.
        expanded acf file: acf_dat + '.' + f'{supercell_size_str}'
    """
    #Functions that are used
    def find_element(list_, line):
        final_result = False
        for element in list_:
            if element in line:
                result = True
            else:
                result = False
            final_result = final_result or result
        return final_result
    
    from pymatgen.io.vasp.inputs import Potcar

    def get_valence_electrons_from_potcar_pymatgen(potcar_file):
        """
        Reads a POTCAR file using pymatgen and extracts the number of valence electrons for each element.

        Args:
            potcar_file (str): Path to the POTCAR file.

        Returns:
            dict: A dictionary where keys are element names and values are the number of valence electrons.
        """
        try:
            # Use pymatgen's Potcar class to parse the POTCAR file
            potcar = Potcar.from_file(potcar_file)

            # Extract element names and their ZVAL values
            valence_electrons = {entry.element: entry.nelectrons for entry in potcar}

            return valence_electrons
        except FileNotFoundError:
            raise FileNotFoundError(f"The file {potcar_file} does not exist.")
        except Exception as e:
            raise RuntimeError(f"An error occurred while parsing the POTCAR file: {e}")

    valence_electrons = get_valence_electrons_from_potcar_pymatgen(potcar_path)
    #print(valence_electrons)
    
    #start
    if len(supercell_size)!= 3:
        return 'supercell size should have this format: [int, int, int]'
    repetition = np.prod(supercell_size) #returns the product of elements
    supercell_size_str = ''.join([str(i) for i in supercell_size]) #to name the file '221'
    new_poscar_path = input_path +  '.' + f'{supercell_size_str}_dummy'
    new_acf_path = acf_dat + '.' + f'{supercell_size_str}'
    new_acf_path_dummy = acf_dat + '.' + f'dummy_{supercell_size_str}'
    new_acf = open(new_acf_path,'w')
    new_acf_dummy = open(new_acf_path_dummy,'w')



    input_ = open(input_path,'r')
    input_lines = input_.readlines()
    input_.close()
    
    #Make a supercell with the new size
    structure = Structure.from_file(input_path)
    structure.make_supercell(supercell_size)
    structure.to(fmt = 'POSCAR',filename = new_poscar_path)
    #Read the expanded poscar
    new_poscar_coordinates = read_CONTCAR(new_poscar_path,fractional=False)

    #Read nonexpanded acf.dat
    acf_dat = open(acf_dat,'r')
    acf_dat_lines = acf_dat.readlines()
    acf_dat.close()
    initial_lines = acf_dat_lines[0:2] #['    #         X           Y           Z        CHARGE     MIN DIST    ATOMIC VOL\n', ' ----------------------------  \n']
    final_lines = acf_dat_lines[-4:] #[' --------------------------------------------------------------------------------\n', '    VACUUM CHARGE: \n', 'VACUUM VOLUME:\n', 'NUMBER OF ELECTRONS:\n']


    #Write the lines of the expanded acf.dat.221
    non_rep_list = ['---','X','VACUUM','NUMBER']
    for line in acf_dat_lines:
        if find_element(non_rep_list, line):
            new_acf_dummy.write(line)
        else:
            for i in range(0, repetition):
                new_acf_dummy.write(line)

    new_acf_dummy.close()

    new_bader_df = bader_df(acf_dat=new_acf_path_dummy,
                            input_type = 'vasp',
                            input_path = new_poscar_path,
                            valence_dict = valence_electrons,
                            slab = True,
                            atom_types_line_num = 5, 
                            type_num_line_num = 6,
                            z_threshhold = 1.0)
    #os.rmtree(new_acf_dummy)

    new_poscar_df = read_CONTCAR(new_poscar_path,output_type='df')

    new_bader_df['x'] = new_poscar_df['x']
    new_bader_df['y'] = new_poscar_df['y']
    new_bader_df['z'] = new_poscar_df['z']
    
    #Writing the final acf file
    col_list = list(new_bader_df.columns)
    #print(col_list)
    all_new_acf_lines = []
    atom_number = [i + 1 for i in list(new_bader_df.index)]
    for row_id, row in new_bader_df.iterrows():
        each_atom_list = ['',
                          str(row_id + 1), 
                          str(round(row[col_list[1]],4)), 
                          str(round(row[col_list[2]],4)), 
                          str(round(row[col_list[3]],4)), 
                          str(round(row[col_list[4]],4)), 
                          str(round(row[col_list[5]],4)),  
                          str(round(row[col_list[6]],4))]
        each_line = '      '.join(each_atom_list)
        all_new_acf_lines.append(each_line) 
        
        
    for line in initial_lines:
        new_acf.write(line)  
    for line in all_new_acf_lines:
        new_acf.write(line + '\n')
    for line in final_lines:
        new_acf.write(line)
        
    new_acf.close()
        
    return new_bader_df
    
        
        
    
     
    
    
    

    
    

In [23]:
# Path to your POTCAR file
my_dir = './test_repeat_structure/0_VASP/SDA_test/3_test/'
potcar_file = my_dir + "POTCAR.surf" 
poscar_file = my_dir + 'POSCAR.surf'
acf_file = my_dir + 'ACF_surf.dat'

expand_charge_file_VASP(acf_file,
                       poscar_file,
                       potcar_file,
                       supercell_size = [2,2,1])


,atom_type,x,y,z,Charge,Min_dist,atomic_volume,surface_atom,bader_charge
0,Cu,2.761198e+00,1.992738,3.393203,10.5224,1.0820,10.5707,0,0.4776
1,Cu,-2.761220e+00,11.557848,3.393203,10.5224,1.0820,10.5707,0,0.4776
2,Cu,1.380604e+01,1.992738,3.393203,10.5224,1.0820,10.5707,0,0.4776
3,Cu,8.283617e+00,11.557848,3.393203,10.5224,1.0820,10.5707,0,0.4776
4,Cu,8.283617e+00,1.992738,3.393203,10.5224,1.0820,10.5707,0,0.4776
5,Cu,2.761199e+00,11.557848,3.393203,10.5224,1.0820,10.5707,0,0.4776
6,Cu,1.932845e+01,1.992738,3.393203,10.5224,1.0820,10.5707,0,0.4776
7,Cu,1.380604e+01,11.557848,3.393203,10.5224,1.0820,10.5707,0,0.4776
8,Cu,-1.086775e-05,6.775293,3.393203,10.5224,1.0820,10.5707,0,0.4776
9,Cu,-5.522429e+00,16.340403,3.393203,10.5224,1.0820,10.5707,0,0.4776


In [24]:
bader_df(acf_file, poscar_file, 
         valence_dict = get_valence_electrons_from_potcar_pymatgen(potcar_file),
         input_type ='vasp')

,atom_type,x,y,z,Charge,Min_dist,atomic_volume,surface_atom,bader_charge
0,Cu,2.7612,1.9927,3.3932,10.5224,1.0820,10.5707,0,0.4776
1,Cu,8.2836,1.9927,3.3932,10.5224,1.0820,10.5707,0,0.4776
2,Cu,0.0000,6.7753,3.3932,10.5224,1.0820,10.5707,0,0.4776
3,Cu,5.5224,6.7753,3.3932,10.5224,1.0820,10.5707,0,0.4776
4,Cu,0.0000,0.3985,8.1227,10.5939,1.1076,47.5525,1,0.4061
5,Cu,5.5224,0.3985,8.1227,10.5939,1.1076,47.5525,1,0.4061
6,Cu,-2.7612,5.1811,8.1227,10.5939,1.1076,47.5525,1,0.4061
7,Cu,2.7612,5.1811,8.1227,10.5939,1.1076,47.5525,1,0.4061
8,Pt,0.0000,0.3985,1.0852,10.0510,1.2429,62.6581,0,-0.0510
9,Pt,2.7612,0.4047,1.1114,10.0813,1.2520,56.8099,0,-0.0813


In [25]:
#Done
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar
import numpy as np

def keep_central_2atmolecule(poscar_file, repetition, molecule_atoms, output_file="POSCAR_central_molecule"):
    """
    Repeats a structure and retains only the central molecule in the supercell, explicitly handling periodic boundaries. 
    However this script only works for biatomic molecules and does not work for N2H or NH3.
    
    Parameters:
        poscar_file (str): Path to the initial POSCAR file.
        repetition (tuple): Repetition factors along x, y, z (e.g., (2, 2, 1)).
        molecule_atoms (list): List of atom types that form the molecule (e.g., ["N"] for N2).
        output_file (str): Path to save the resulting POSCAR file (default: "POSCAR_central_molecule").
    
    Returns:
        list of numpy arrays: Coordinates of the central molecule in fractional format.
    """
    # Load the initial structure

    structure = Structure.from_file(poscar_file)

    # Repeat the structure
    scaling_matrix = [[repetition[0], 0, 0],
                      [0, repetition[1], 0],
                      [0, 0, repetition[2]]]
    supercell = structure * scaling_matrix

    # Identify the geometric center of the supercell
    cell_center = np.array([0.5, 0.5, 0.5])  # Center in fractional coordinates

    # Find indices of atoms belonging to the specified molecule
    molecule_indices = [i for i, site in enumerate(supercell) if site.species_string in molecule_atoms]
    print(molecule_indices)
    # Pair atoms in the molecule (assuming they are bonded), explicitly handling periodic boundaries
    molecule_pairs = []
    cutoff_distance = 1.5  # Typical bond distance, adjust if needed
    used_indices = set()

    for i in molecule_indices:
        if i in used_indices:
            continue
        for j in molecule_indices:
            if j > i and j not in used_indices:
                # Calculate fractional distance, considering periodic boundaries
                frac_diff = supercell[j].frac_coords - supercell[i].frac_coords
                frac_diff -= np.round(frac_diff)  # Adjust for PBC
                dist = np.linalg.norm(np.dot(frac_diff, supercell.lattice.matrix))
                if dist <= cutoff_distance:
                    molecule_pairs.append((i, j))
                    used_indices.update([i, j])
                    break

    # Find the molecule closest to the geometric center
    def pair_distance_to_center(pair):
        atom1_coords = supercell[pair[0]].frac_coords
        atom2_coords = supercell[pair[1]].frac_coords
        # Correct fractional coordinates for PBC
        frac_diff = atom2_coords - atom1_coords
        frac_diff -= np.round(frac_diff)  # Adjust for PBC
        atom2_coords_corrected = atom1_coords + frac_diff
        center_of_mass = (atom1_coords + atom2_coords_corrected) / 2
        # Ensure center_of_mass is within the unit cell
        center_of_mass %= 1.0
        return np.linalg.norm(center_of_mass - cell_center)

    central_pair = min(molecule_pairs, key=pair_distance_to_center)

    # Extract coordinates of the central molecule
    atom1_coords = supercell[central_pair[0]].frac_coords
    atom2_coords = supercell[central_pair[1]].frac_coords
    frac_diff = atom2_coords - atom1_coords
    frac_diff -= np.round(frac_diff)  # Adjust for PBC
    atom2_coords_corrected = atom1_coords + frac_diff
    central_molecule_coords = [atom1_coords, atom2_coords_corrected % 1.0]

    # Create a new structure retaining only the central molecule
    central_structure = supercell.copy()
    central_structure.remove_sites([index for pair in molecule_pairs if pair != central_pair for index in pair])

    # Save the resulting structure
    Poscar(central_structure).write_file(output_file)
    print(f"Central molecule saved to {output_file}")

    # Return the central molecule's fractional coordinates
    return central_molecule_coords


In [26]:
#TEST two atom mol
mol_list = ['N2',"H2",
           # 'N2H',"NH3"
           ]
def atom_types(mol_name):
    my_list = []
    for i in mol_name:
        if i == 'N' or i == 'H':
            my_list.append(i)
    return my_list
for mol in mol_list:
    input_path = f'./test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.s5{mol}'
    output_path = f'./test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.s5{mol}_expanded_centralmol'
    keep_central_2atmolecule(
        poscar_file=input_path,  # Input POSCAR file
        repetition=(2, 2, 1),  # Repetition factors
        molecule_atoms=atom_types(mol),  # Atoms forming the molecule (e.g., N2)
        output_file=output_path # Output file
    )

[256, 257, 258, 259, 260, 261, 262, 263]
Central molecule saved to ./test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.s5N2_expanded_centralmol
[256, 257, 258, 259, 260, 261, 262, 263]
Central molecule saved to ./test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.s5H2_expanded_centralmol


In [27]:
#Done
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar
import numpy as np

def keep_central_3_4atmolecule(poscar_file, repetition, molecule_atoms, output_file="POSCAR_central_molecule"):
    """
    Repeats a structure and retains only the central molecule in the supercell, explicitly handling periodic boundaries.
    Supports multi-atom molecules such as N2H or NH3 but 'not' biatomic molecule.
    
    Parameters:
        poscar_file (str): Path to the initial POSCAR file.
        repetition (tuple): Repetition factors along x, y, z (e.g., (2, 2, 1)).
        molecule_atoms (list): List of atom types that form the molecule (e.g., ["N", "H"] for NH3).
        output_file (str): Path to save the resulting POSCAR file (default: "POSCAR_central_molecule").
    
    Returns:
        list of numpy arrays: Coordinates of the central molecule in fractional format.
    """
    # Load the initial structure
    structure = Structure.from_file(poscar_file)

    # Repeat the structure
    scaling_matrix = [[repetition[0], 0, 0],
                      [0, repetition[1], 0],
                      [0, 0, repetition[2]]]
    supercell = structure * scaling_matrix

    # Identify the geometric center of the supercell
    cell_center = np.array([0.5, 0.5, 0.5])  # Center in fractional coordinates

    # Find indices of atoms belonging to the specified molecule
    molecule_indices = [i for i, site in enumerate(supercell) if site.species_string in molecule_atoms]

    # Group atoms into molecules based on a bonding cutoff distance
    cutoff_distance = 1.5  # Adjust as needed
    molecule_groups = []
    visited_indices = set()

    for i in molecule_indices:
        if i in visited_indices:
            continue
        group = [i]
        visited_indices.add(i)
        # Find all bonded atoms in the molecule
        for j in molecule_indices:
            if j not in visited_indices:
                for atom_index in group:
                    # Calculate fractional distance, considering PBC
                    frac_diff = supercell[j].frac_coords - supercell[atom_index].frac_coords
                    frac_diff -= np.round(frac_diff)  # Adjust for PBC
                    dist = np.linalg.norm(np.dot(frac_diff, supercell.lattice.matrix))
                    if dist <= cutoff_distance:
                        group.append(j)
                        visited_indices.add(j)
                        break
        molecule_groups.append(group)

    # Find the molecule closest to the geometric center
    def group_distance_to_center(group):
        # Calculate the center of mass of the group
        frac_coords = [supercell[i].frac_coords for i in group]
        frac_coords = np.array(frac_coords)
        center_of_mass = frac_coords.mean(axis=0) % 1.0
        return np.linalg.norm(center_of_mass - cell_center)

    central_group = min(molecule_groups, key=group_distance_to_center)

    # Extract coordinates of the central molecule
    central_molecule_coords = [supercell[i].frac_coords for i in central_group]

    # Create a new structure retaining only the central molecule
    central_structure = supercell.copy()
    central_structure.remove_sites([i for group in molecule_groups if group != central_group for i in group])

    # Save the resulting structure
    Poscar(central_structure).write_file(output_file)
    print(f"Central molecule saved to {output_file}")

    # Return the central molecule's fractional coordinates
    return central_molecule_coords



In [28]:
#TEST 3_4 atom molecule
mol_list = [
           # 'N2',"H2",
            'N2H',
   # "NH3"
           ]
def atom_types(mol_name):
    my_list = []
    for i in mol_name:
        if i == 'N' or i == 'H':
            my_list.append(i)
    return my_list
for mol in mol_list:
    input_path = f'./test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.s8{mol}'
    output_path = f'./test_repeat_structure/0_VASP/SDA_test/POSCAR.s8{mol}_expanded_centralmol'
    mol_coord = keep_central_3_4atmolecule(
        poscar_file=input_path,  # Input POSCAR file
        repetition=(2, 2, 1),  # Repetition factors
        molecule_atoms=atom_types(mol),  # Atoms forming the molecule (e.g., N2)
        output_file=output_path # Output file
    )

Central molecule saved to ./test_repeat_structure/0_VASP/SDA_test/POSCAR.s8N2H_expanded_centralmol


In [29]:
#Done
#mol = "N2H"
#surface_pos = f'./test_repeat_structure/0_VASP/SDA_test/POSCAR.surf221'
#dummy_pos =   f'./test_repeat_structure/0_VASP/SDA_test/POSCAR.s8{mol}_expanded_centralmol'
#out_file = f'./test_repeat_structure/0_VASP/SDA_test/POSCAR.s8{mol}_final_expanded_centralmol'
#mol_atoms = atom_types(mol)


def add_molecule_to_surface(dummy_poscar_path, 
                            main_poscar_path, 
                            output_poscar_path, 
                            molecule_species):
    """
    Adds molecule coordinates from the dummy POSCAR to the main POSCAR based on the specified species.
    
    Parameters:
    - dummy_poscar_path (str): Path to the dummy POSCAR file containing the surface + molecule.
    - main_poscar_path (str): Path to the main POSCAR file containing only the surface we want to use.
    - output_poscar_path (str): Path to save the updated POSCAR file.
    - molecule_species (list): List of species names for the molecule (e.g., ['N', 'H']).
    """
    def find_coordinate_start(poscar_lines):
        """
        Determines the starting line for coordinates in a POSCAR file.

        Parameters:
        - poscar_lines (list): List of lines from a POSCAR file.

        Returns:
        - int: The starting line number for coordinates (either 8 or 9).
        """
        
        
        # Check if line 7 contains the "Selective dynamics" keyword
        if poscar_lines[7].strip().lower() == "selective dynamics":
            return 9  # Coordinates start on line 9
        return 8  # Coordinates start on line 8
    
    
    # Read both POSCAR files
    with open(dummy_poscar_path, 'r') as f:
        dummy_poscar = f.readlines()
    with open(main_poscar_path, 'r') as f:
        main_poscar = f.readlines()
    

    # Extract species and atom counts from the dummy POSCAR
    dummy_species = dummy_poscar[5].strip().split()
    print('dummy species',dummy_species)
    dummy_atom_counts = list(map(int, dummy_poscar[6].strip().split()))
    print('dummy counts', dummy_atom_counts)
    
    # Calculate the starting line for coordinates in the dummy POSCAR
    total_surface_atoms = sum(dummy_atom_counts[:len(main_poscar[5].strip().split())])  # Surface atom count
    total_atom_counts = sum(dummy_atom_counts)
    print('dummy number of surface atoms',total_surface_atoms)
    coordinates_start = find_coordinate_start(dummy_poscar)  # Coordinates start after species and atom count lines
    coordinates = dummy_poscar[coordinates_start: coordinates_start + total_atom_counts]
    #print('dummy last coordinate',coordinates)

    # Extract molecule coordinates based on the provided species
    molecule_coordinates = []
    for species in molecule_species:
        index = dummy_species.index(species)
        count = dummy_atom_counts[index]
        count_atoms_before = 0
        for n in range(0, index):
            count_atoms_before = count_atoms_before + dummy_atom_counts[n]
        start_mol_coord = count_atoms_before
        line_num = coordinates[count_atoms_before:count_atoms_before + count]
        molecule_coordinates.extend(line_num)
    print('mol_coords',molecule_coordinates)

    # Update species and atom counts in the main POSCAR
    main_species = main_poscar[5].strip().split()
    print('main_species',main_species)
    
    main_atom_counts = list(map(int, main_poscar[6].strip().split()))
    print('main_atom_counts',main_atom_counts)

    updated_species = main_species + molecule_species
    print('updated_species',updated_species)
    updated_atom_counts = main_atom_counts + [dummy_atom_counts[dummy_species.index(species)] for species in molecule_species]
    print('updated_atom_counts',updated_atom_counts)
    # Write the updated POSCAR
    with open(output_poscar_path, 'w') as f:
        # Write the unchanged header and lattice information
        f.writelines(main_poscar[:5])
        # Write the updated species and atom counts
        f.write(" ".join(updated_species) + "\n")
        f.write(" ".join(map(str, updated_atom_counts)) + "\n")
        # Write the rest of the main POSCAR
        f.writelines(main_poscar[7:])
        # Append molecule coordinates in the order of species
        f.writelines(molecule_coordinates)


In [30]:
mol = 'N2H'
surface_pos = f'./test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.surf221'
dummy_pos =   f'./test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.s8{mol}_expanded_centralmol'
out_file = f'./test_repeat_structure/0_VASP/SDA_test/0_test/POSCAR.s8{mol}_final_expanded_centralmol'
mol_atoms = atom_types(mol)

add_molecule_to_surface(dummy_pos, 
                        surface_pos, 
                        out_file, 
                        mol_atoms)

dummy species ['Co', 'Ni', 'N', 'H']
dummy counts [192, 64, 2, 1]
dummy number of surface atoms 256
mol_coords ['0.583429 0.479638 0.384931 T T T N\n', '0.529737 0.489771 0.389697 T T T N\n', '0.512933 0.491065 0.437131 T T T H\n']
main_species ['Co', 'Ni']
main_atom_counts [192, 64]
updated_species ['Co', 'Ni', 'N', 'H']
updated_atom_counts [192, 64, 2, 1]


In [31]:
mol_list = ['N2','H2','N2H','NH3']
test_numbers = [1, 2, 3, 4, 5, 6]
for test_number in test_numbers:
    test_dir = f'./test_repeat_structure/0_VASP/SDA_test/{test_number}_test/'
    surf_input = test_dir + 'POSCAR.surf'
    surf_acf = test_dir + 'ACF_surf.dat'

    for mol in mol_list:
        mol_atoms = atom_types(mol)
        mol_surf_dummy = test_dir + f'POSCAR.{mol}'
        expand_struct_vasp(surf_input, 
                       surf_input + '221',
                           [2,2,1])

        if len(mol_atoms) == 1:
            keep_central_2atmolecule(mol_surf_dummy,
                                    [2,2,1],
                                    mol_atoms,
                                    output_file= mol_surf_dummy + '_cent221')
        else:
            keep_central_3_4atmolecule(mol_surf_dummy,
                                    [2,2,1],
                                    mol_atoms,
                                    output_file= mol_surf_dummy + '_cent221')   

        add_molecule_to_surface(mol_surf_dummy + '_cent221',
                                surf_input + '221',
                                mol_surf_dummy + '_final',
                                mol_atoms)


    

Structure is Expanded
[336, 337, 338, 339, 340, 341, 342, 343]
Central molecule saved to ./test_repeat_structure/0_VASP/SDA_test/1_test/POSCAR.N2_cent221
dummy species ['Pt', 'V', 'N']
dummy counts [296, 40, 2]
dummy number of surface atoms 336
mol_coords ['0.662380 0.663912 0.407636 T T T N\n', '0.681266 0.702721 0.407636 T T T N\n']
main_species ['Pt', 'V']
main_atom_counts [296, 40]
updated_species ['Pt', 'V', 'N']
updated_atom_counts [296, 40, 2]
Structure is Expanded
[336, 337, 338, 339, 340, 341, 342, 343]
Central molecule saved to ./test_repeat_structure/0_VASP/SDA_test/1_test/POSCAR.H2_cent221
dummy species ['Pt', 'V', 'H']
dummy counts [296, 40, 2]
dummy number of surface atoms 336
mol_coords ['0.662379 0.671350 0.407636 T T T H\n', '0.681268 0.695282 0.407636 T T T H\n']
main_species ['Pt', 'V']
main_atom_counts [296, 40]
updated_species ['Pt', 'V', 'H']
updated_atom_counts [296, 40, 2]
Structure is Expanded
Central molecule saved to ./test_repeat_structure/0_VASP/SDA_test/1_

[288, 289, 290, 291, 292, 293, 294, 295]
Central molecule saved to ./test_repeat_structure/0_VASP/SDA_test/6_test/POSCAR.N2_cent221
dummy species ['Ni', 'Zn', 'N']
dummy counts [144, 144, 2]
dummy number of surface atoms 288
mol_coords ['0.448163 0.661475 0.356978 T T T N\n', '0.468504 0.703545 0.356978 T T T N\n']
main_species ['Ni', 'Zn']
main_atom_counts [144, 144]
updated_species ['Ni', 'Zn', 'N']
updated_atom_counts [144, 144, 2]
Structure is Expanded
[288, 289, 290, 291, 292, 293, 294, 295]
Central molecule saved to ./test_repeat_structure/0_VASP/SDA_test/6_test/POSCAR.H2_cent221
dummy species ['Ni', 'Zn', 'H']
dummy counts [144, 144, 2]
dummy number of surface atoms 288
mol_coords ['0.448163 0.669539 0.356978 T T T H\n', '0.468504 0.695482 0.356978 T T T H\n']
main_species ['Ni', 'Zn']
main_atom_counts [144, 144]
updated_species ['Ni', 'Zn', 'H']
updated_atom_counts [144, 144, 2]
Structure is Expanded
Central molecule saved to ./test_repeat_structure/0_VASP/SDA_test/6_test/POSCA

In [32]:
mol_list = ['H2']

test_dir = './test_repeat_structure/0_VASP/SDA_test/1_test/'
surf_input = test_dir + 'POSCAR.surf'
surf_acf = test_dir + 'ACF_surf.dat'

for mol in mol_list:
    mol_atoms = atom_types(mol)
    mol_surf_dummy = test_dir + f'POSCAR.{mol}'
    expand_struct_vasp(surf_input, 
                    surf_input + '221',
                        [2,2,1])

    if len(mol_atoms) == 1:
        keep_central_2atmolecule(mol_surf_dummy,
                                [2,2,1],
                                 mol_atoms,
                                output_file= mol_surf_dummy + '_cent221')
    else:
        keep_central_3_4atmolecule(mol_surf_dummy,
                                 [2,2,1],
                                 mol_atoms,
                                  output_file= mol_surf_dummy + '_cent221')   

    add_molecule_to_surface(mol_surf_dummy + '_cent221',
                                surf_input + '221',
                                mol_surf_dummy + '_final',
                                mol_atoms)


    

Structure is Expanded
[336, 337, 338, 339, 340, 341, 342, 343]
Central molecule saved to ./test_repeat_structure/0_VASP/SDA_test/1_test/POSCAR.H2_cent221
dummy species ['Pt', 'V', 'H']
dummy counts [296, 40, 2]
dummy number of surface atoms 336
mol_coords ['0.662379 0.671350 0.407636 T T T H\n', '0.681268 0.695282 0.407636 T T T H\n']
main_species ['Pt', 'V']
main_atom_counts [296, 40]
updated_species ['Pt', 'V', 'H']
updated_atom_counts [296, 40, 2]


In [33]:
#Done
def calculate_nearest_slab_distances_with_atom_info(poscar_file, molecule_atom_types, num_neighbor = 9):
    """
    Calculates the distances of the 9 nearest atoms in the slab from the molecule center.
    Includes the atom type, Cartesian coordinates, distance vector, distance magnitude, 
    and the order of each atom in the POSCAR file.

    Args:
        poscar_file (str): Path to the POSCAR file.
        molecule_atom_types (list): List of molecule atom types to identify (e.g., ['N', 'H']).
        num_neighbor (int): Number of nearest neighbors to return. Default is 9.

    Returns:
        dict: A dictionary containing:
            - 'molecule_center': Cartesian coordinates of the molecule center.
            - 'nearest_atoms': A list of the 9 nearest atoms with the following info:
                - Atom type (e.g., 'Pt', 'N').
                - Cartesian coordinates as a tuple.
                - Distance vector (relative position from molecule center).
                - Distance from the molecule center (magnitude of distance vector).
                - Atom order in the POSCAR file (0-based index).
    """
    import numpy as np
    from pymatgen.io.vasp import Poscar

    # Load POSCAR file and extract structure
    with open(poscar_file, "r") as f:
        file_content = f.read()  # Read file contents to ensure it's accessible

    poscar = Poscar.from_string(file_content)
    structure = poscar.structure

    # Convert fractional coordinates to Cartesian coordinates
    lattice_matrix = structure.lattice.matrix  # Lattice vectors (3x3 matrix)
    fractional_coords = structure.frac_coords  # Fractional coordinates of all atoms
    all_coords = np.dot(fractional_coords, lattice_matrix)  # Convert fractional to Cartesian
    all_atom_types = structure.species  # List of atom types

    # Filter molecule atoms by type
    molecule_coords = []
    molecule_indices = []
    for i, atom_type in enumerate(all_atom_types):
        if str(atom_type) in molecule_atom_types:
            molecule_coords.append(all_coords[i])
            molecule_indices.append(i)

    if len(molecule_coords) == 0:
        raise ValueError(f"No atoms of the specified types {molecule_atom_types} were found in the POSCAR file.")

    molecule_coords = np.array(molecule_coords)

    # Explicitly extract coordinates of N atoms
    nitrogen_coords = [
        all_coords[i] for i, atom in enumerate(all_atom_types) if str(atom) == "N"
    ]

    # Determine the molecule center based on the specified strategy
    if len(nitrogen_coords) == 2:
        # Two N atoms: Use the center of the two N atoms
        molecule_center = np.mean(nitrogen_coords, axis=0)
    elif len(nitrogen_coords) == 1:
        # One N atom: Use its coordinate
        molecule_center = nitrogen_coords[0]
    elif len(molecule_coords) == 2:
        # Biatomic molecule (not N₂): Use the center of the two atoms
        molecule_center = np.mean(molecule_coords, axis=0)
    else:
        # No N atoms or other multi-atom molecule: Use the center of all atoms
        molecule_center = np.mean(molecule_coords, axis=0)

    # Calculate distances from the molecule center to all slab atoms
    distances = []
    slab_atoms_info = []
    for i, coord in enumerate(all_coords):
        # Exclude distances between molecule atoms themselves
        if i in molecule_indices:
            continue
        distance_vector = coord - molecule_center  # Compute the distance vector
        distance = np.linalg.norm(distance_vector)  # Compute the magnitude of the distance vector
        distances.append(distance)
        slab_atoms_info.append((str(all_atom_types[i]), tuple(coord), tuple(distance_vector), distance, i))  # 0-based index

    # Sort by distance and select the nearest num_neighbor atoms
    sorted_atoms_info = sorted(slab_atoms_info, key=lambda x: x[3])[:num_neighbor]

    # Prepare results
    result = {
        "molecule_center": molecule_center.tolist(),
        "nearest_atoms": sorted_atoms_info  # Each entry contains (type, coordinates, distance vector, distance magnitude, index)
    }
    return result


In [34]:
#poscar_path = "./test_repeat_structure/0_VASP/SDA_test/1_test/POSCAR.H2" #Path to your POSCAR file
poscar_path = './test_repeat_structure/0_VASP/SDA_test/1_test/POSCAR.N2'
molecule_atom_types = ['N']

nearest_neighbors = calculate_nearest_slab_distances_with_atom_info(poscar_path, molecule_atom_types)

    
print("Molecule Center:", nearest_neighbors["molecule_center"])
print("Nearest Atoms:")
for atom_info in nearest_neighbors["nearest_atoms"]:
    atom_type, cart_coords, distance_vector, distance, index = atom_info
    print(f"Atom Type: {atom_type}, Coordinates: {cart_coords}, Distance Vector: {distance_vector}, Distance: {distance:.4f}, Index: {index}")

Molecule Center: [3.029476982326, 4.55092436129, 10.009467400633]
Nearest Atoms:
Atom Type: Pt, Coordinates: (3.0293248816219998, 4.507075888378001, 9.110537072088), Distance Vector: (-0.00015210070400017983, -0.043848472911999004, -0.8989303285449992), Distance: 0.9000, Index: 32
Atom Type: V, Coordinates: (4.408881208269001, 6.5491031860260005, 8.861883413339001), Distance Vector: (1.3794042259430013, 1.9981788247360006, -1.1475839872939986), Distance: 2.6856, Index: 79
Atom Type: Pt, Coordinates: (1.810991392582, 6.807587204494999, 9.131917774188), Distance Vector: (-1.218485589744, 2.2566628432049995, -0.8775496264449991), Distance: 2.7106, Index: 38
Atom Type: Pt, Coordinates: (5.786270698226, 4.504703040688, 9.116916771313), Distance Vector: (2.7567937158999998, -0.04622132060200013, -0.8925506293199987), Distance: 2.8981, Index: 33
Atom Type: Pt, Coordinates: (-0.001239820004, 4.782967095896, 9.047936796776), Distance Vector: (-3.0307168023299997, 0.2320427346059999, -0.96153060

In [35]:
#Done
def get_nearest_neighbors_with_bader_and_elneg(poscar_file, molecule_atom_types, acf_surf, poscar_surf, potcar_surf, supercell_size=[2,2,1], num_neighbor=9):
    """
    Finds the nearest neighbors of a molecule in a slab, retrieves their Bader charge, and calculates their electronegativity.

    Args:
        poscar_file (str): Path to the POSCAR file containing the slab and molecule (result from add_molecule func).
        molecule_atom_types (list): List of molecule atom types to identify (e.g., ['N', 'H']).
        acf_surf (str): Path to the unexpanded acf.dat file of the surface.
        poscar_surf (str): Path to the unexpanded POSCAR file of the surface.
        potcar_surf (str): Path to the POTCAR file of the surface.
        supercell_size (list, optional): Supercell expansion size. Default is [2,2,1].
        num_neighbor (int, optional): Number of nearest neighbors to find. Default is 9.

    Returns:
        dict: A dictionary containing:
            - 'molecule_center': Cartesian coordinates of the molecule center.
            - 'nearest_atoms': List of dictionaries for each nearest atom containing:
                - Atom type
                - Cartesian coordinates
                - Distance vector
                - Distance magnitude
                - Atom index in POSCAR
                - Bader charge
                - Electronegativity
    """
    import numpy as np

    # Step 1: Get nearest neighbors using the first function
    nearest_neighbors_data = calculate_nearest_slab_distances_with_atom_info(poscar_file, molecule_atom_types, num_neighbor)

    # Step 2: Expand charge file and get Bader charge data
    bader_df = expand_charge_file_VASP(acf_surf, poscar_surf, potcar_surf, supercell_size=supercell_size)

    # Step 3: Retrieve Bader charge and electronegativity for each nearest neighbor
    nearest_neighbors_with_extra_info = []
    
    for atom_info in nearest_neighbors_data["nearest_atoms"]:
        atom_type, cart_coords, distance_vector, distance, atom_index = atom_info

        # Fetch the Bader charge from the dataframe using the atom index
        try:
            bader_charge = bader_df.loc[atom_index, "bader_charge"]  # Adjust column name if needed
        except KeyError:
            bader_charge = None  # If index is not found, return None

        # Get the electronegativity for the atom type
        try:
            electronegativity = get_elneg(atom_type)
        except:
            electronegativity = None  # If not found, return None

        # Append all gathered info
        nearest_neighbors_with_extra_info.append({
            "atom_type": atom_type,
            "coordinates": cart_coords,
            "distance_vector": distance_vector,
            "distance": distance,
            "index": atom_index,
            "bader_charge": bader_charge,
            "electronegativity": electronegativity
        })

    # Step 4: Prepare final output dictionary
    result = {
        "molecule_center": nearest_neighbors_data["molecule_center"],
        "nearest_atoms": nearest_neighbors_with_extra_info
    }

    return result


In [36]:
# Define file paths
test_number = 2
poscar_file = f"./test_repeat_structure/0_VASP/SDA_test/{test_number}_test/POSCAR.N2_final"  # POSCAR file for the molecule
acf_surf = f"./test_repeat_structure/0_VASP/SDA_test/{test_number}_test/ACF_surf.dat"  # Bader charge file of the surface
poscar_surf = f"./test_repeat_structure/0_VASP/SDA_test/{test_number}_test/POSCAR.surf"  # POSCAR file of the surface
potcar_surf = f"./test_repeat_structure/0_VASP/SDA_test/{test_number}_test/POTCAR.surf"  # POTCAR file of the surface

# Define molecule atom types
molecule_atom_types = ["N",
                      #"H"
                      ]  # Example: Nitrogen and Hydrogen atoms

# Define supercell size
supercell_size = [2,2,1]  # Expanding in x2, y2, z1

# Define number of nearest neighbors
num_neighbors = 9

# Run the function
result = get_nearest_neighbors_with_bader_and_elneg(
    poscar_file=poscar_file,
    molecule_atom_types=molecule_atom_types,
    acf_surf=acf_surf,
    poscar_surf=poscar_surf,
    potcar_surf=potcar_surf,
    supercell_size=supercell_size,
    num_neighbor=num_neighbors
)

# Print results
print("\nMolecule Center:", result["molecule_center"])
print("\nNearest Atoms with Bader Charges and Electronegativity:")
for atom in result["nearest_atoms"]:
    print(f"Atom Type: {atom['atom_type']}")
    print(f"  Coordinates: {atom['coordinates']}")
    print(f"  Distance Vector: {atom['distance_vector']}")
    print(f"  Distance: {atom['distance']:.4f}")
    print(f"  Atom Index: {atom['index']}")
    print(f"  Bader Charge: {atom['bader_charge']}")
    print(f"  Electronegativity: {atom['electronegativity']}\n")
result


Molecule Center: [8.8763826438045, 12.416967730585, 7.716119081476]

Nearest Atoms with Bader Charges and Electronegativity:
Atom Type: Ni
  Coordinates: (8.876390837391, 12.41695501025, 6.816119322389)
  Distance Vector: (8.193586499061212e-06, -1.2720335000437899e-05, -0.8999997590869997)
  Distance: 0.9000
  Atom Index: 110
  Bader Charge: -0.22460000000000058
  Electronegativity: 1.91

Atom Type: Zn
  Coordinates: (10.241983125, 14.53631546527, 7.228378128166001)
  Distance Vector: (1.3656004811955, 2.1193477346850003, -0.48774095330999945)
  Distance: 2.5680
  Atom Index: 255
  Bader Charge: 0.15970000000000084
  Electronegativity: 1.65

Atom Type: Zn
  Coordinates: (7.510782162609, 14.53631546527, 7.228378128166001)
  Distance Vector: (-1.365600481195501, 2.1193477346850003, -0.48774095330999945)
  Distance: 2.5680
  Atom Index: 261
  Bader Charge: 0.15970000000000084
  Electronegativity: 1.65

Atom Type: Zn
  Coordinates: (10.241983125, 10.29619531838, 7.228378128166001)
  Dist

{'molecule_center': [8.8763826438045, 12.416967730585, 7.716119081476],
 'nearest_atoms': [{'atom_type': 'Ni',
   'coordinates': (8.876390837391, 12.41695501025, 6.816119322389),
   'distance_vector': (8.193586499061212e-06,
    -1.2720335000437899e-05,
    -0.8999997590869997),
   'distance': 0.8999997592141896,
   'index': 110,
   'bader_charge': -0.22460000000000058,
   'electronegativity': 1.91},
  {'atom_type': 'Zn',
   'coordinates': (10.241983125, 14.53631546527, 7.228378128166001),
   'distance_vector': (1.3656004811955,
    2.1193477346850003,
    -0.48774095330999945),
   'distance': 2.5679545814308264,
   'index': 255,
   'bader_charge': 0.15970000000000084,
   'electronegativity': 1.65},
  {'atom_type': 'Zn',
   'coordinates': (7.510782162609, 14.53631546527, 7.228378128166001),
   'distance_vector': (-1.365600481195501,
    2.1193477346850003,
    -0.48774095330999945),
   'distance': 2.567954581430827,
   'index': 261,
   'bader_charge': 0.15970000000000084,
   'electrone

# OUTCAR Files 

In [37]:
def extract_total_energy_from_outcar(outcar_file):
    """
    Extracts the last total energy (TOTEN) from a VASP OUTCAR file.

    Args:
        outcar_file (str): Path to the OUTCAR file.

    Returns:
        float: Last total energy in eV, or None if not found.
    """
    last_energy = None  # Initialize to None
    
    try:
        with open(outcar_file, "r") as f:
            for line in f:
                if "free  energy   TOTEN" in line:
                    last_energy = float(line.split()[-2])  # Extract the energy value

    except FileNotFoundError:
        print(f"Error: The file {outcar_file} does not exist.")
    except Exception as e:
        print(f"Error reading OUTCAR file: {e}")

    return last_energy  # Return the last found energy value


In [38]:
outcar_path = './test_repeat_structure/0_VASP/SDA_test_outcar/OUTCAR'
extract_total_energy_from_outcar(outcar_path)

-421.47162546

In [39]:
from pymatgen.io.vasp import Poscar
from pymatgen.core.structure import Structure

def calculate_molecule_distances(poscar_file, molecule_atoms):
    """
    Reads a CONTCAR file, extracts a molecular structure, and calculates distances between the molecule's atoms,
    considering periodic boundary conditions (PBC).

    Args:
        poscar_file (str): Path to the CONTCAR file.
        molecule_atoms (list): List of atomic symbols representing the molecule (e.g., ["N", "H"] for NH3).

    Returns:
        list: A sorted list of calculated distances based on the molecule type.
    """
    try:
        # Read CONTCAR file as a string
        with open(poscar_file, "r") as f:
            file_content = f.read()

        # Use Poscar.from_string() to parse the file content
        poscar = Poscar.from_string(file_content)
        structure = poscar.structure

        # Extract only molecule atoms
        molecule_sites = [site for site in structure.sites if site.species_string in molecule_atoms]

        if len(molecule_sites) < 2:
            raise ValueError("Molecule must have at least two atoms to compute distances.")

        distances = []

        if len(molecule_sites) == 2:
            # Biatomic case (e.g., N2, O2)
            distances.append(structure.lattice.get_distance_and_image(
                molecule_sites[0].frac_coords, molecule_sites[1].frac_coords
            )[0])

        elif len(molecule_sites) == 3:
            # Case: N2H (2N, 1H) → Find N-N and smallest N-H distance
            nitrogen_atoms = [site for site in molecule_sites if site.species_string == "N"]
            hydrogen_atoms = [site for site in molecule_sites if site.species_string == "H"]

            if len(nitrogen_atoms) == 2 and len(hydrogen_atoms) == 1:
                nn_distance = structure.lattice.get_distance_and_image(
                    nitrogen_atoms[0].frac_coords, nitrogen_atoms[1].frac_coords
                )[0]

                nh_distances = [
                    structure.lattice.get_distance_and_image(nitrogen.frac_coords, hydrogen_atoms[0].frac_coords)[0]
                    for nitrogen in nitrogen_atoms
                ]

                distances.append(nn_distance)
                distances.append(min(nh_distances))  # Choose the smallest N-H distance

        elif len(molecule_sites) == 4:
            # Case: NH3 (1N, 3H) → Compute 3 N-H distances and sort them
            nitrogen_atoms = [site for site in molecule_sites if site.species_string == "N"]
            hydrogen_atoms = [site for site in molecule_sites if site.species_string == "H"]

            if len(nitrogen_atoms) == 1 and len(hydrogen_atoms) == 3:
                nh_distances = [
                    structure.lattice.get_distance_and_image(nitrogen_atoms[0].frac_coords, h.frac_coords)[0]
                    for h in hydrogen_atoms
                ]

                distances.extend(sorted(nh_distances))  # Sort from smallest to largest

        return distances

    except Exception as e:
        print(f"Error processing CONTCAR file: {e}")
        return None


In [40]:
# Define the path to your CONTCAR file

# Example 0: Biatomic molecule (e.g., N2)
number_list = list(range(1,10))


molecule_atoms = ["N"]
for i in number_list:
    contcar_path = f'./test_repeat_structure/0_VASP/SDA_test_outcar/N2/CONTCAR.N2_{i}'
    distances = calculate_molecule_distances(contcar_path, molecule_atoms)
    print(f"N-N distance CONTCAR.N2_{i}:", distances)

# Example 1: Biatomic molecule (e.g., H2)
molecule_atoms = ["H"]
for i in number_list:
    contcar_path = f'./test_repeat_structure/0_VASP/SDA_test_outcar/H2/CONTCAR.H2_{i}'
    distances = calculate_molecule_distances(contcar_path, molecule_atoms)
    print(f"H-H distance CONTCAR.H2_{i}:", distances)

# Example 2: N2H molecule (2N, 1H)
molecule_atoms = ["N", "H"]
for i in number_list:
    contcar_path = f'./test_repeat_structure/0_VASP/SDA_test_outcar/N2H/CONTCAR.N2H_{i}'
    distances = calculate_molecule_distances(contcar_path, molecule_atoms)
    print(f"N-N and smaller N-H distance CONTCAR.N2H_{i}:", distances)

# Example 3: NH3 molecule (1N, 3H)
molecule_atoms = ["N", "H"]
for i in number_list:
    contcar_path = f'./test_repeat_structure/0_VASP/SDA_test_outcar/NH3/CONTCAR.NH3_{i}'
    distances = calculate_molecule_distances(contcar_path, molecule_atoms)
    print(f"Sorted N-H distances CONTCAR.NH3_{i}:", distances)


N-N distance CONTCAR.N2_1: [1.120982238881245]
N-N distance CONTCAR.N2_2: [4.850461155034299]
N-N distance CONTCAR.N2_3: [3.239021168379713]
N-N distance CONTCAR.N2_4: [3.991119522183379]
N-N distance CONTCAR.N2_5: [1.1209822388812458]
N-N distance CONTCAR.N2_6: [1.120982238881246]
N-N distance CONTCAR.N2_7: [7.511014953007913]
N-N distance CONTCAR.N2_8: [5.878624279688735]
N-N distance CONTCAR.N2_9: [1.1209822388812447]
H-H distance CONTCAR.H2_1: [0.7393934951642832]
H-H distance CONTCAR.H2_2: [4.125070970552226]
H-H distance CONTCAR.H2_3: [3.3032620850380603]
H-H distance CONTCAR.H2_4: [3.4054371245753647]
H-H distance CONTCAR.H2_5: [0.7393934951642832]
H-H distance CONTCAR.H2_6: [0.7393934951642832]
H-H distance CONTCAR.H2_7: [5.994769241221559]
H-H distance CONTCAR.H2_8: [4.009793909177997]
H-H distance CONTCAR.H2_9: [0.7393934951642829]
N-N and smaller N-H distance CONTCAR.N2H_1: [1.1779757259171637, 1.1443160199114117]
N-N and smaller N-H distance CONTCAR.N2H_2: [2.65924711616424

In [41]:
#Dont forget for each poscar H2 N2 consider the electronegativity of molecule atoms 
# Do some testing on the merged function
# Find out the status of mol in outcar: distances of mol atoms from each other, distance from surface 

In [42]:
def outcar_converged(outcar_file):
    """
    Reads an OUTCAR file, checks if the convergence criteria is met.

    Args:
        outcar_file (str): Path to the CONTCAR file.

    Returns:
        int: A value of 0 (not converged structure) or 1 (converged).
    """
    converged = 0
    with open(outcar_file, "r") as f:
        outcar_lines = f.readlines()
    for line in outcar_lines:
        if 'reached required accuracy - stopping structural energy minimisation' in line:
            converged = converged + 1
    return converged


In [43]:
outcar_converged('./test_repeat_structure/0_VASP/SDA_test_outcar/OUTCAR')

1

# Copying the files ready

In [8]:
#Copy bader files
import os
import shutil
Subhmoy_materials_str = 'Co3Ni  CoPt3  CuPt   Fe3Co   FeCo  FeNi3  FePt3  NiPt VPt   VPt3  Zn11Co2  Zn13Co  Zn22Ni3 Zn3Cu  Zn8Cu5  ZnCu  ZnPt Co3Pt  Cu3Pt  CuPt7  Fe9Co7  FeNi  FePt   Ni3Pt  NiPt3  V3Pt VPt2  VPt8  Zn11Ni2  Zn13Fe  Zn35Cu17  Zn3Pt  Zn9Fe4  ZnNi  ZnPt3'
Subhmoy_materials = Subhmoy_materials_str.split()

Satvik_materials_str = 'Al13Fe4  Al3Ni   Al3Pt5  Al5Co2  AlCu3  AlNi3  AlV   V3Ni   VFe3 Al21Pt8  Al3Ni2  Al3V    Al9Co2   AlFe   AlPt   AlV3  V3Ni2  VNi2 Al2Cu    Al3Ni5  Al4Cu9  AlCo    AlFe3  AlPt2  V3Co  V4Zn5  VNi3 Al2Pt    Al3Pt2  Al4Ni3  AlCu    AlNi   AlPt3  V3Fe  VCo3   VZn3'
Satvik_materials = Satvik_materials_str.split()

mols = ['N2','H2','N2H','NH3']

person ='Subhmoy'
if person == 'Subhmoy':
    used_materials = Subhmoy_materials
if person == 'Satvik':
    used_materials = Satvik_materials
    
origin_bader = f'/run/user/1001/gvfs/smb-share:server=10.50.16.153,share=data-iml/Parastoo/big_ad_sites_project/scf_bader_DOS_surf/{person}/'
destination = f'/home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/{person}/'

for i in range(len(used_materials)):
    material = str(i) + '_' + used_materials[i]
    origin_mat_path = origin_bader + f'{material}/1_surf/postprocess/1_bader/ACF.dat'
    dest_mat_path = destination + f'{material}/1_surf/'
    if os.path.exists(origin_mat_path):
        shutil.copy(origin_mat_path, dest_mat_path)
    else:
        print(f'File {person}/{material}/ACF.dat does NOT exist')

In [48]:
#Copy adsorption files
#working on now
import os
import shutil

Subhmoy_materials_str = 'Co3Ni  CoPt3  CuPt   Fe3Co   FeCo  FeNi3  FePt3  NiPt VPt   VPt3  Zn11Co2  Zn13Co  Zn22Ni3 Zn3Cu  Zn8Cu5  ZnCu  ZnPt Co3Pt  Cu3Pt  CuPt7  Fe9Co7  FeNi  FePt   Ni3Pt  NiPt3  V3Pt VPt2  VPt8  Zn11Ni2  Zn13Fe  Zn35Cu17  Zn3Pt  Zn9Fe4  ZnNi  ZnPt3'
Subhmoy_materials = Subhmoy_materials_str.split()

Satvik_materials_str = 'Al13Fe4  Al3Ni   Al3Pt5  Al5Co2  AlCu3  AlNi3  AlV   V3Ni   VFe3 Al21Pt8  Al3Ni2  Al3V    Al9Co2   AlFe   AlPt   AlV3  V3Ni2  VNi2 Al2Cu    Al3Ni5  Al4Cu9  AlCo    AlFe3  AlPt2  V3Co  V4Zn5  VNi3 Al2Pt    Al3Pt2  Al4Ni3  AlCu    AlNi   AlPt3  V3Fe  VCo3   VZn3'
Satvik_materials = Satvik_materials_str.split()

mols = ['N2','H2','N2H','NH3']

person ='Satvik'
if person == 'Subhmoy':
    used_materials = Subhmoy_materials
if person == 'Satvik':
    used_materials = Satvik_materials
    
copy_files = ['CONTCAR','POSCAR_initial', 'OUTCAR']
    

destination = f'/home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/{person}/'

origin = f'/run/user/1001/gvfs/smb-share:server=10.50.16.153,share=data-iml/Shared_ML/{person}_cedar/'

def poscar_initial_origin(mol, person, material, adnum):
    if mol == '2_N2' or mol == '3_H2':
        return f'/home/parastoo/Desktop/ML_project/{person}/{material}/{mol}/{adnum}/'
    
    elif mol == '4_N2H':
        return f'/home/parastoo/Desktop/ML_project/N2H/{person}/{material}/{mol}/{adnum}/'
    
    elif mol == '5_NH3':
        return f'/home/parastoo/Desktop/ML_project/NH3/{person}/{material}/{mol}/{adnum}/'
    
    

for i in range(len(used_materials)):
#for i in range(0,1):
    material = str(i) + '_' + used_materials[i]
    material_origin_dir = origin + f"{material}/"
    #for m in range(len(mols)):
    #for m in range(0,4):
    for m in range(3,4):
        mol = str(m+2) + '_' + mols[m]
        mol_dir = material_origin_dir + f'{mol}/'
        if os.path.isdir(mol_dir):
            ad_sites_dirs = []
            for each_dir in os.listdir(mol_dir):
                if 'ad_site' in each_dir:
                    ad_sites_dirs.append(each_dir)
            ad_sites_dirs = sorted(ad_sites_dirs,key=lambda x: int(re.findall(r'\d+', x)[0]))
            for ad_site_num in ad_sites_dirs:
                ad_site_path = mol_dir + f'{ad_site_num}/'
                outcar_adsite_path = ad_site_path + 'OUTCAR'
                contcar_adsite_path = ad_site_path + 'CONTCAR'
                if os.path.exists(outcar_adsite_path) and outcar_converged(outcar_adsite_path):
                    os.makedirs(destination + f'{material}/{mol}/{ad_site_num}/',exist_ok=True)
                    shutil.copy(ad_site_path + 'OUTCAR',destination + f'{material}/{mol}/{ad_site_num}/')
                    if os.path.exists(contcar_adsite_path):
                        shutil.copy(ad_site_path + 'CONTCAR',destination + f'{material}/{mol}/{ad_site_num}/')
                    else:
                        print(f'CONTCAR does not exist:   {contcar_adsite_path}')
                    if os.path.exists(ad_site_path + 'POSCAR_initial'):
                        shutil.copy(ad_site_path + 'POSCAR_initial',destination + f'{material}/{mol}/{ad_site_num}/')
                    else:
                        poscar_initial_path = poscar_initial_origin(mol, person, material, ad_site_num) + 'POSCAR_initial'
                        shutil.copy(poscar_initial_path, destination + f'{material}/{mol}/{ad_site_num}/')
                            
            
            
        else:
            f'{person}/{material}/{mol_dir} does NOT exist'
        
    
    
    

# Expansions

In [50]:
#We want to do the expansions of mol+surface POSCAR initial ones
import os
import shutil

Subhmoy_materials_str = 'Co3Ni  CoPt3  CuPt   Fe3Co   FeCo  FeNi3  FePt3  NiPt VPt   VPt3  Zn11Co2  Zn13Co  Zn22Ni3 Zn3Cu  Zn8Cu5  ZnCu  ZnPt Co3Pt  Cu3Pt  CuPt7  Fe9Co7  FeNi  FePt   Ni3Pt  NiPt3  V3Pt VPt2  VPt8  Zn11Ni2  Zn13Fe  Zn35Cu17  Zn3Pt  Zn9Fe4  ZnNi  ZnPt3'
Subhmoy_materials = Subhmoy_materials_str.split()

Satvik_materials_str = 'Al13Fe4  Al3Ni   Al3Pt5  Al5Co2  AlCu3  AlNi3  AlV   V3Ni   VFe3 Al21Pt8  Al3Ni2  Al3V    Al9Co2   AlFe   AlPt   AlV3  V3Ni2  VNi2 Al2Cu    Al3Ni5  Al4Cu9  AlCo    AlFe3  AlPt2  V3Co  V4Zn5  VNi3 Al2Pt    Al3Pt2  Al4Ni3  AlCu    AlNi   AlPt3  V3Fe  VCo3   VZn3'
Satvik_materials = Satvik_materials_str.split()

mols = ['N2','H2','N2H','NH3']

mols_dict = {'N2':['N'],
            'H2':['H'],
            'N2H':['N','H'],
            'NH3':['N','H'],}

person ='Subhmoy'
if person == 'Subhmoy':
    used_materials = Subhmoy_materials
if person == 'Satvik':
    used_materials = Satvik_materials
    
origin = f'/home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/{person}/'



for i in range(len(used_materials)):
#for i in range(0,1):
    material = str(i) + '_' + used_materials[i]
    material_origin_dir = origin + f"{material}/"
    surf_input = material_origin_dir + '1_surf/POSCAR.surf441'
    #for m in range(len(mols)):
    for m in range(3,4):
        mol_atom_type =  mols_dict[mols[m]]
        mol = str(m+2) + '_' + mols[m]
        mol_dir = material_origin_dir + f'{mol}/'
        if len(os.listdir(mol_dir)) > 0:
            ad_sites_dirs = []
            for each_dir in os.listdir(mol_dir):
                if 'ad_site' in each_dir:
                    ad_sites_dirs.append(each_dir)
            ad_sites_dirs = sorted(ad_sites_dirs,key=lambda x: int(re.findall(r'\d+', x)[0]))
            for ad_site_num in ad_sites_dirs:
                ad_site_path = mol_dir + f'{ad_site_num}/'
                if len(mol_atom_type) == 1:
                    keep_central_2atmolecule(ad_site_path + 'POSCAR_initial',
                                            [4,4,1],
                                             mol_atom_type,
                                            output_file= ad_site_path + 'POSCAR_initial' + '_cent441')
                else:
                    keep_central_3_4atmolecule(ad_site_path + 'POSCAR_initial',
                                            [4,4,1],
                                             mol_atom_type,
                                            output_file= ad_site_path + 'POSCAR_initial' + '_cent441')   

                add_molecule_to_surface(ad_site_path + 'POSCAR_initial' + '_cent441',
                                            surf_input,
                                            ad_site_path + 'POSCAR_initial' + '_final441',
                                            mol_atom_type)                
                

Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/0_Co3Ni/5_NH3/1_ad_site/POSCAR_initial_cent441
dummy species ['Co', 'Ni', 'N', 'H']
dummy counts [768, 256, 1, 3]
dummy number of surface atoms 1024
mol_coords ['0.521353 0.510747 0.384611 T T T N\n', '0.507014 0.522120 0.402525 T T T H\n', '0.547100 0.525092 0.402549 T T T H\n', '0.509977 0.485005 0.402528 T T T H\n']
main_species ['Co', 'Ni']
main_atom_counts [768, 256]
updated_species ['Co', 'Ni', 'N', 'H']
updated_atom_counts [768, 256, 1, 3]
Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/0_Co3Ni/5_NH3/2_ad_site/POSCAR_initial_cent441
dummy species ['Co', 'Ni', 'N', 'H']
dummy counts [768, 256, 1, 3]
dummy number of surface atoms 1024
mol_coords ['0.520787 0.572963 0.385570 T T T N\n', '0.506447 0.584337 0.403484 T T T H\n', '0.546534 0.587309 0.403508 T T T H\n', '0.509411 0.547222 0.403486 T T T H\n']
main_species ['C

Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/2_CuPt/5_NH3/1_ad_site/POSCAR_initial_cent441
dummy species ['Cu', 'Pt', 'N', 'H']
dummy counts [1024, 768, 1, 3]
dummy number of surface atoms 1792
mol_coords ['0.005208 0.510417 0.358279 T T T N\n', '0.996136 0.517613 0.377007 T T T H\n', '0.021500 0.519494 0.377032 T T T H\n', '0.998011 0.494129 0.377010 T T T H\n']
main_species ['Cu', 'Pt']
main_atom_counts [1024, 768]
updated_species ['Cu', 'Pt', 'N', 'H']
updated_atom_counts [1024, 768, 1, 3]
Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/2_CuPt/5_NH3/2_ad_site/POSCAR_initial_cent441
dummy species ['Cu', 'Pt', 'N', 'H']
dummy counts [1024, 768, 1, 3]
dummy number of surface atoms 1792
mol_coords ['0.598958 0.572917 0.358279 T T T N\n', '0.589885 0.580113 0.377007 T T T H\n', '0.615250 0.581994 0.377032 T T T H\n', '0.591761 0.556629 0.377010 T T T H\n']
main_species [

Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/4_FeCo/5_NH3/6_ad_site/POSCAR_initial_cent441
dummy species ['Co', 'Fe', 'N', 'H']
dummy counts [576, 576, 1, 3]
dummy number of surface atoms 1152
mol_coords ['0.520834 0.444446 0.463703 T T T N\n', '0.497535 0.452549 0.487939 T T T H\n', '0.542444 0.454666 0.487972 T T T H\n', '0.522574 0.426106 0.487943 T T T H\n']
main_species ['Co', 'Fe']
main_atom_counts [576, 576]
updated_species ['Co', 'Fe', 'N', 'H']
updated_atom_counts [576, 576, 1, 3]
Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/4_FeCo/5_NH3/7_ad_site/POSCAR_initial_cent441
dummy species ['Co', 'Fe', 'N', 'H']
dummy counts [576, 576, 1, 3]
dummy number of surface atoms 1152
mol_coords ['0.562500 0.430557 0.461649 T T T N\n', '0.539202 0.438660 0.485886 T T T H\n', '0.584110 0.440777 0.485918 T T T H\n', '0.564240 0.412217 0.485889 T T T H\n']
main_species ['Co'

Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/6_FePt3/5_NH3/8_ad_site/POSCAR_initial_cent441
dummy species ['Fe', 'Pt', 'N', 'H']
dummy counts [256, 768, 1, 3]
dummy number of surface atoms 1024
mol_coords ['0.531025 0.531136 0.391433 T T T N\n', '0.518110 0.541380 0.409347 T T T H\n', '0.554217 0.544057 0.409371 T T T H\n', '0.520779 0.507949 0.409350 T T T H\n']
main_species ['Fe', 'Pt']
main_atom_counts [256, 768]
updated_species ['Fe', 'Pt', 'N', 'H']
updated_atom_counts [256, 768, 1, 3]
Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/6_FePt3/5_NH3/9_ad_site/POSCAR_initial_cent441
dummy species ['Fe', 'Pt', 'N', 'H']
dummy counts [256, 768, 1, 3]
dummy number of surface atoms 1024
mol_coords ['0.531252 0.593749 0.392181 T T T N\n', '0.518336 0.603994 0.410095 T T T H\n', '0.554444 0.606671 0.410119 T T T H\n', '0.521006 0.570562 0.410098 T T T H\n']
main_species ['F

Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/10_Zn11Co2/5_NH3/5_ad_site/POSCAR_initial_cent441
dummy species ['Co', 'Zn', 'N', 'H']
dummy counts [160, 1024, 1, 3]
dummy number of surface atoms 1184
mol_coords ['0.575660 0.484221 0.402099 T T T N\n', '0.553586 0.491898 0.419266 T T T H\n', '0.596134 0.493904 0.419289 T T T H\n', '0.577308 0.466845 0.419269 T T T H\n']
main_species ['Co', 'Zn']
main_atom_counts [160, 1024]
updated_species ['Co', 'Zn', 'N', 'H']
updated_atom_counts [160, 1024, 1, 3]
Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/10_Zn11Co2/5_NH3/6_ad_site/POSCAR_initial_cent441
dummy species ['Co', 'Zn', 'N', 'H']
dummy counts [160, 1024, 1, 3]
dummy number of surface atoms 1184
mol_coords ['0.388160 0.421720 0.402099 T T T N\n', '0.366086 0.429398 0.419266 T T T H\n', '0.408634 0.431404 0.419289 T T T H\n', '0.389808 0.404345 0.419269 T T T H\n']
main_s

...

Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/27_VPt8/5_NH3/11_ad_site/POSCAR_initial_cent441
dummy species ['Pt', 'V', 'N', 'H']
dummy counts [1184, 160, 1, 3]
dummy number of surface atoms 1344
mol_coords ['0.375021 0.425084 0.403679 T T T N\n', '0.352616 0.432864 0.420114 T T T H\n', '0.395808 0.434708 0.420136 T T T H\n', '0.376701 0.409855 0.420116 T T T H\n']
main_species ['Pt', 'V']
main_atom_counts [1184, 160]
updated_species ['Pt', 'V', 'N', 'H']
updated_atom_counts [1184, 160, 1, 3]
Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/27_VPt8/5_NH3/12_ad_site/POSCAR_initial_cent441
dummy species ['Pt', 'V', 'N', 'H']
dummy counts [1184, 160, 1, 3]
dummy number of surface atoms 1344
mol_coords ['0.588210 0.427558 0.409301 T T T N\n', '0.565805 0.435339 0.425735 T T T H\n', '0.608997 0.437183 0.425757 T T T H\n', '0.589890 0.412329 0.425738 T T T H\n']
main_species [

Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/28_Zn11Ni2/5_NH3/4_ad_site/POSCAR_initial_cent441
dummy species ['Ni', 'Zn', 'N', 'H']
dummy counts [128, 736, 1, 3]
dummy number of surface atoms 864
mol_coords ['0.504984 0.513315 0.347956 T T T N\n', '0.482833 0.521019 0.366685 T T T H\n', '0.525530 0.523032 0.366710 T T T H\n', '0.506638 0.495878 0.366687 T T T H\n']
main_species ['Ni', 'Zn']
main_atom_counts [128, 736]
updated_species ['Ni', 'Zn', 'N', 'H']
updated_atom_counts [128, 736, 1, 3]
Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/28_Zn11Ni2/5_NH3/5_ad_site/POSCAR_initial_cent441
dummy species ['Ni', 'Zn', 'N', 'H']
dummy counts [128, 736, 1, 3]
dummy number of surface atoms 864
mol_coords ['0.613213 0.427276 0.331579 T T T N\n', '0.591062 0.434980 0.350308 T T T H\n', '0.633760 0.436993 0.350333 T T T H\n', '0.614868 0.409839 0.350311 T T T H\n']
main_species

Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/31_Zn3Pt/5_NH3/3_ad_site/POSCAR_initial_cent441
dummy species ['Pt', 'Zn', 'N', 'H']
dummy counts [512, 1536, 1, 3]
dummy number of surface atoms 2048
mol_coords ['0.546591 0.534953 0.365284 T T T N\n', '0.527984 0.539952 0.384013 T T T H\n', '0.563849 0.541258 0.384038 T T T H\n', '0.547980 0.523639 0.384016 T T T H\n']
main_species ['Pt', 'Zn']
main_atom_counts [512, 1536]
updated_species ['Pt', 'Zn', 'N', 'H']
updated_atom_counts [512, 1536, 1, 3]
Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/31_Zn3Pt/5_NH3/4_ad_site/POSCAR_initial_cent441
dummy species ['Pt', 'Zn', 'N', 'H']
dummy counts [512, 1536, 1, 3]
dummy number of surface atoms 2048
mol_coords ['0.562358 0.582361 0.367398 T T T N\n', '0.543751 0.587360 0.386127 T T T H\n', '0.579616 0.588667 0.386152 T T T H\n', '0.563747 0.571047 0.386129 T T T H\n']
main_speci

Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/32_Zn9Fe4/5_NH3/2_ad_site/POSCAR_initial_cent441
dummy species ['Fe', 'Zn', 'N', 'H']
dummy counts [320, 896, 1, 3]
dummy number of surface atoms 1216
mol_coords ['0.509857 0.502796 0.430831 T T T N\n', '0.487656 0.510517 0.447312 T T T H\n', '0.530448 0.512535 0.447334 T T T H\n', '0.511514 0.485321 0.447314 T T T H\n']
main_species ['Fe', 'Zn']
main_atom_counts [320, 896]
updated_species ['Fe', 'Zn', 'N', 'H']
updated_atom_counts [320, 896, 1, 3]
Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/32_Zn9Fe4/5_NH3/3_ad_site/POSCAR_initial_cent441
dummy species ['Fe', 'Zn', 'N', 'H']
dummy counts [320, 896, 1, 3]
dummy number of surface atoms 1216
mol_coords ['0.447356 0.596888 0.430831 T T T N\n', '0.425156 0.604608 0.447312 T T T H\n', '0.467948 0.606626 0.447334 T T T H\n', '0.449014 0.579412 0.447314 T T T H\n']
main_species

Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/33_ZnNi/5_NH3/7_ad_site/POSCAR_initial_cent441
dummy species ['Ni', 'Zn', 'N', 'H']
dummy counts [576, 576, 1, 3]
dummy number of surface atoms 1152
mol_coords ['0.604167 0.466250 0.363225 T T T N\n', '0.580041 0.473894 0.381953 T T T H\n', '0.626544 0.475891 0.381979 T T T H\n', '0.605968 0.448951 0.381956 T T T H\n']
main_species ['Ni', 'Zn']
main_atom_counts [576, 576]
updated_species ['Ni', 'Zn', 'N', 'H']
updated_atom_counts [576, 576, 1, 3]
Central molecule saved to /home/parastoo/Desktop/ML_project/analysis/ML_project/transferred_files/Subhmoy/33_ZnNi/5_NH3/8_ad_site/POSCAR_initial_cent441
dummy species ['Ni', 'Zn', 'N', 'H']
dummy counts [576, 576, 1, 3]
dummy number of surface atoms 1152
mol_coords ['0.604167 0.438473 0.363225 T T T N\n', '0.580041 0.446116 0.381953 T T T H\n', '0.626544 0.448113 0.381979 T T T H\n', '0.605968 0.421174 0.381956 T T T H\n']
main_species ['N